In [ ]:
# There are 2 types of Agents - 
"""
1. Simple Agent:
    The agent enters a loop where it sends a request and checks whether the response contains a tool call.
    If so, it executes the function (e.g. get_weather) and appends the result to the conversation.
    The loop stops when there are no more tool calls and the response contains final text (output_text).
2. Objective-Based Agent:
    The agent is given a custom objective function (e.g. check whether the phrase "task complete" is in the output).
    It loops until the objective function returns True.
-------------------------------------
https://github.com/BrightPool/udemy-prompt-engineering-course/blob/main/openai_features_and_functionality/learning_agents_from_scratch.ipynb
-------------------------------------
Agent 1 (Simple Agent):
Uses a while loop to repeatedly call the Responses API.
Checks for tool (function) calls, executes them, and appends the output to the conversation.
The loop stops when a final response (output_text) is provided and there are no more tool calls.

Agent 2 (Objective-Based Agent):
Uses a custom objective function (objective_met) to decide when to stop the loop.
Continues to gather responses and execute tools until the agent's output includes a key phrase ("task complete").
------------
get_capital(country) → calls a country API and returns Paris
get_weather(city) → calls a geocoding API to convert Paris → latitude/longitude, then calls the weather API using those coordinates.

"""
import os, sys
sys.path.append(r"C:\Users\allan\projects\Prompt_Engineering")
from llm_config import MODEL_GROQ, groq_api_key 
from getpass import getpass
import requests
import json
import requests
from dotenv import load_dotenv
load_dotenv(override=True)
capital_key = os.getenv("capital_key")

In [3]:
from os import name
def get_capital(countrycode):
    countrycode = 'CA'
    response = requests.get(
        f"https://api.restcountries.com/countries/v5/codes.alpha_2/{countrycode}?pretty=1/",
        headers={
            "Authorization": capital_key
        }
    )
    data = response.json()
    return data['data']['objects'][0]['capitals'][0]['name'] 

In [4]:
get_capital("CA")

'Ottawa'

In [5]:
# Convert city → latitude/longitude
import requests
def get_weather(city):
    geo_response = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={
            "name": city,
            "count": 1,
            "language": "en",
            "format": "json"
        }
    ) 
    geo_data = geo_response.json()
    latitude = geo_data["results"][0]["latitude"]
    longitude = geo_data["results"][0]["longitude"]

    weather_response = requests.get(
        f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m")
    data = weather_response.json()
    return data['current']['temperature_2m']


In [ ]:
get_weather('Paris') 

15.6

In [15]:
# This agent sends a prompt (asking about the weather), then enters an agentic loop.
# At each turn, it calls the Responses API:

# 1. If the response contains a tool call: The agent executes the function (using our get_weather tool) and 
# appends the function result to the conversation as a new message.
# 2. If the response provides output text: The agent stops, printing the final output.
"""
User
 │
 │ "weather in capital of France"
 ▼
LLM
 │
 │ First needs capital
 ▼
get_capital(country="France")
 │
 ▼
"Paris"
 │
 ▼
LLM
 │
 │ Now needs Paris coordinates
 ▼
get_weather(latitude=48.8566, longitude=2.3522)
 │
 ▼
Weather
 │
 ▼
Final answer
"""
# from llm_config import groq, groq_url
from ast import arguments
from Agents import tool_schema
from llm_config import ollama, MODEL_OLLAMA
from Agents.tool_schema import tool_schema_def

messages = [
    {
        "role": "system",
        "content": """
        For weather questions asking about the capital of a country Canada which is arbrivated form to CA. Donot take that as california from USA:
        1. First call get_capital to find the capital city.
        2. After receiving the capital, call get_weather using that city.
        3. Do not call get_weather before get_capital.
        """
    },
    {
        "role": "user",
        "content": "What's the weather in the capital of CA today?"
    }
]

max_iteration = 5
iteration = 0 
while iteration < max_iteration:
    iteration += 1
    response = ollama.responses.create(
        model = MODEL_OLLAMA, 
        input = messages,
        tools = tool_schema_def,
    )
    messages.extend(response.output)           #  # preserve LLM's tool call

    list_of_all_functions = [
        all_tools for all_tools in response.output
        if hasattr(all_tools, 'type') and all_tools.type == "function_call"     # hasattr(object, "attribute")
    ]
    for (num, fc) in enumerate(list_of_all_functions , 1):  #Go through each function call, give it a number starting from 1, and print its name and arguments.
        print(f"{num}, {fc},{fc.arguments}")

# Process all function calls from this iteration
    for fc_item in list_of_all_functions:
        if fc_item.name == "get_capital":
            args = json.loads(fc_item.arguments)
            fc_result = get_capital(args['country'])
        elif fc_item.name == "get_weather":                       
            args = json.loads(fc_item.arguments)
            fc_result = get_weather(args['city'])
        
        messages.append({                           # model needs to know which tool call the result belongs to.
            "type" : "function_call_output",
            "call_id" : fc_item.call_id,
            "output" : fc_result,
            }) 

# If no function calls, we have the final answer
    if not list_of_all_functions:
        print(response.output_text)
        break



1, ResponseFunctionToolCall(arguments='{"country":"Canada"}', call_id='call_4ynyzh5r', name='get_capital', type='function_call', id='fc_resp_519491_0', status='completed'),{"country":"Canada"}
The current weather in Ottawa, Canada.

"The current temperature is 14 degrees Celsius and there is a light breeze. The humidity is 60%, and the sky is mostly cloudy."
